# Notebook 05: Embeddings and ChromaDB Integration

**Project:** Advance AI Research Assistant  
**Module:** Document Processing and Vector Database  
**Notebook:** `05_embeddings_and_chromadb.ipynb`  
**Status:** In Progress

---

## 1. Overview

This notebook implements the embedding generation and vector storage components of the Advance AI Research Assistant.

The preprocessed and chunked scientific documents from Notebook 04 are converted into numerical vector representations (embeddings) using a Sentence Transformer model. These embeddings are then stored in a persistent ChromaDB collection to enable semantic search and information retrieval.

This module establishes the foundation for the Retrieval-Augmented Generation (RAG) pipeline.

## 2. Objectives

The main objectives of this notebook are:

- Load the processed document chunks from the previous module.
- Initialize a lightweight Sentence Transformer embedding model.
- Generate vector embeddings for all document chunks.
- Create and configure a persistent ChromaDB collection.
- Store embeddings together with their document metadata.
- Verify the stored vectors and metadata.
- Perform semantic similarity searches to retrieve relevant document passages.

## 3. Tools and Technologies

| Component | Technology |
|---|---|
| Programming language | Python 3.11 |
| Embedding framework | Sentence Transformers |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector database | ChromaDB |
| Numerical processing | NumPy |
| Data analysis | Pandas |
| Deep learning framework | PyTorch |

## 4. Input and Expected Output

**Input:**
- `data/processed/document_chunks.jsonl`

**Expected outputs:**
- Generated text embeddings (384 dimensions).
- Persistent vector database in `data/chroma_db/`.
- ChromaDB collection: `research_papers`.
- Semantic search results for user queries.
- `data/processed/semantic_search_test.csv`

## 5. Implementation Workflow

The notebook follows these steps:

1. Load project dependencies and configuration.
2. Read the processed document chunks.
3. Load the embedding model.
4. Generate embeddings for all chunks.
5. Initialize the persistent ChromaDB database.
6. Store vectors and associated metadata.
7. Validate the stored records.
8. Test semantic search.
9. Export a search-results report.

## 6. Expected Outcome

At the end of this notebook, the application should have a persistent vector database containing the embeddings of all processed document chunks. The database should support semantic similarity search while preserving the source document and page information required for later citation generation.

This completes the embedding and vector-storage stage and prepares the project for hybrid information retrieval and the RAG pipeline.

In [2]:
# Imports and project paths
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import chromadb

from sentence_transformers import SentenceTransformer

# Project paths
PROJECT_ROOT = (
    Path.home()
    / "Schreibtisch"
    / "IfaDW"
    / "NLP"
    / "Final_Project"
    / "advance-ai-research-assistant"
)

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed" / "document_chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

assert CHUNKS_FILE.exists(), f"Chunk file not found: {CHUNKS_FILE}"

print("Project:", PROJECT_ROOT)
print("Input:", CHUNKS_FILE)
print("ChromaDB directory:", CHROMA_DIR)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Project: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant
Input: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/processed/document_chunks.jsonl
ChromaDB directory: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/chroma_db
PyTorch: 2.14.1+cu130
CUDA available: True


In [3]:
# Load the processed chunks
chunks = []

with CHUNKS_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            chunks.append(json.loads(line))

print("Chunks loaded:", len(chunks))

assert len(chunks) == 156
assert all(chunk["text"].strip() for chunk in chunks)

chunks_df = pd.DataFrame(chunks)

display(
    chunks_df[
        ["chunk_id", "filename", "page_number", "word_count"]
    ].head()
)

Chunks loaded: 156


,chunk_id,filename,page_number,word_count
0,41598_2019_Article_38798_p1_c1,41598_2019_Article_38798.pdf,1,174
1,41598_2019_Article_38798_p1_c2,41598_2019_Article_38798.pdf,1,193
2,41598_2019_Article_38798_p1_c3,41598_2019_Article_38798.pdf,1,185
3,41598_2019_Article_38798_p1_c4,41598_2019_Article_38798.pdf,1,160
4,41598_2019_Article_38798_p1_c5,41598_2019_Article_38798.pdf,1,24


In [4]:
# Load the embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading model:", MODEL_NAME)
print("Selected device:", DEVICE)

embedding_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded successfully.")
print("Embedding dimensions:", embedding_model.get_sentence_embedding_dimension())

Loading model: sentence-transformers/all-MiniLM-L6-v2
Selected device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.
Embedding dimensions: 384


/tmp/ipykernel_34849/1701907434.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", embedding_model.get_sentence_embedding_dimension())


In [5]:
# Generate embeddings
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding generation completed.")
print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding generation completed.
Embedding shape: (156, 384)
Data type: float32


In [6]:
# Validate the embeddings:
assert embeddings.shape == (len(chunks), 384)
assert np.isfinite(embeddings).all()

print("All embeddings are valid.")

All embeddings are valid.


In [7]:
# Create the persistent ChromaDB database
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

collection = chroma_client.get_or_create_collection(
    name="research_papers",
    metadata={
        "hnsw:space": "cosine",
        "description": "Scientific research paper chunks"
    }
)

print("ChromaDB initialized.")
print("Collection:", collection.name)
print("Existing vectors:", collection.count())

ChromaDB initialized.
Collection: research_papers
Existing vectors: 0


In [8]:
# Store chunks, embeddings, and metadata
ids = [chunk["chunk_id"] for chunk in chunks]

metadatas = [
    {
        "document_id": chunk["document_id"],
        "filename": chunk["filename"],
        "page_number": int(chunk["page_number"]),
        "chunk_index": int(chunk["chunk_index"]),
        "word_count": int(chunk["word_count"])
    }
    for chunk in chunks
]

collection.upsert(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas
)

print("Chunks stored successfully.")
print("Vectors in collection:", collection.count())

Chunks stored successfully.
Vectors in collection: 156


In [9]:
# Verify the stored data
stored_data = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

for i in range(len(stored_data["ids"])):
    print("=" * 80)
    print("Chunk ID:", stored_data["ids"][i])
    print("Filename:", stored_data["metadatas"][i]["filename"])
    print("Page:", stored_data["metadatas"][i]["page_number"])
    print("Text preview:", stored_data["documents"][i][:300])

Chunk ID: 41598_2019_Article_38798_p1_c1
Filename: 41598_2019_Article_38798.pdf
Page: 1
Text preview: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seye
Chunk ID: 41598_2019_Article_38798_p1_c2
Filename: 41598_2019_Article_38798.pdf
Page: 1
Text preview: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to technical replicates of the same experiment. Although we use an unsupervised approa
Chunk ID: 41598_2019_Article_38798_p1_c3
Filename: 41598_2019_Article_38798.pdf
Page: 1
Text preview: uripotent stem cells (iPSCs), to some specific tissue required for a patient, such as neuron

In [10]:
# Check the total number of stored vectors:
total_vectors = collection.count()

assert total_vectors == len(chunks)

print("Expected vectors:", len(chunks))
print("Stored vectors:", total_vectors)
print("ChromaDB validation successful.")

Expected vectors: 156
Stored vectors: 156
ChromaDB validation successful.


In [11]:
# Test semantic search
query = "How do deep neural networks help identify different cell types?"

query_embedding = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

search_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

for i in range(len(search_results["ids"][0])):
    metadata = search_results["metadatas"][0][i]

    print("=" * 80)
    print(f"Result {i + 1}")
    print("Filename:", metadata["filename"])
    print("Page:", metadata["page_number"])
    print("Distance:", round(search_results["distances"][0][i], 4))
    print("Text:", search_results["documents"][0][i][:500])

Result 1
Filename: 41598_2019_Article_38798.pdf
Page: 1
Distance: 0.3625
Text: www.nature.com/scientificreports OPEN Cell Identity Codes: Understanding Cell Identity from Gene Expression Profiles using Deep Neural Received: 5 September 2017Accepted: 10 January 2019 Networks Published: xx xx xxxx Farzad Abdolhosseini1, Behrooz Azarkhalili2, Abbas Maazallahi1, Aryan Kamal1, Seyed Abolfazl Motahari1, Ali Sharifi-Zarchi 1 & Hamidreza Chitsaz3 Understanding cell identity is an important task in many biomedical areas. Expression patterns of specific marker genes have been used t
Result 2
Filename: 41598_2019_Article_38798.pdf
Page: 1
Distance: 0.3923
Text: ific structure of deep autoencoders that can encode a GEP into a vector of 30 numeric values, which we call the cell identity code (CIC). The original GEP can be reproduced from the CIC with an accuracy comparable to technical replicates of the same experiment. Although we use an unsupervised approach to train the autoencoder, we show diff

In [12]:
# Save a small search-results report
results_report = []

for i in range(len(search_results["ids"][0])):
    metadata = search_results["metadatas"][0][i]

    results_report.append({
        "rank": i + 1,
        "chunk_id": search_results["ids"][0][i],
        "filename": metadata["filename"],
        "page_number": metadata["page_number"],
        "distance": search_results["distances"][0][i],
        "text_preview": search_results["documents"][0][i][:300]
    })

results_df = pd.DataFrame(results_report)

display(results_df)

REPORT_FILE = PROJECT_ROOT / "data" / "processed" / "semantic_search_test.csv"

results_df.to_csv(REPORT_FILE, index=False, encoding="utf-8")

print("Search report saved:", REPORT_FILE)

,rank,chunk_id,filename,page_number,distance,text_preview
0,1,41598_2019_Article_38798_p1_c1,41598_2019_Article_38798.pdf,1,0.362497,www.nature.com/scientificreports OPEN Cell Ide...
1,2,41598_2019_Article_38798_p1_c2,41598_2019_Article_38798.pdf,1,0.392345,ific structure of deep autoencoders that can e...
2,3,41598_2019_Article_38798_p5_c6,41598_2019_Article_38798.pdf,5,0.392351,"cell type, while other CICs of the same sample..."
3,4,41598_2019_Article_38798_p2_c1,41598_2019_Article_38798.pdf,2,0.455514,www.nature.com/scientificreports/ Kullback-Lei...
4,5,s41598-019-52937-5_p2_c4,s41598-019-52937-5.pdf,2,0.463819,"NN) based classifier, uses somatic point mutat..."


Search report saved: /home/r/Schreibtisch/IfaDW/NLP/Final_Project/advance-ai-research-assistant/data/processed/semantic_search_test.csv


## 7. Conclusion

The embedding generation and vector storage pipeline was successfully implemented and tested.

The sentence-transformers/all-MiniLM-L6-v2 model generated 384-dimensional embeddings for all 156 document chunks. These embeddings were stored in a persistent ChromaDB collection named research_papers, together with their corresponding document metadata.

A semantic search experiment demonstrated that the system can retrieve relevant passages from the indexed scientific papers based on the meaning of a natural-language query.

The successful implementation establishes the vector retrieval component required for the next stage of the Advance AI Research Assistant.

Key results:

156 document chunks embedded.

384-dimensional vector representations generated.

Persistent ChromaDB collection created.

Document and page metadata preserved.

Semantic search successfully tested.